# Scam Call Shield — detector training on AMD GPU (ROCm)

Trains both HUMAN-vs-AI voice detectors on an **AMD Instinct GPU** via ROCm PyTorch:
1. CNN baseline (`voice_classifier/train.py`)
2. **wav2vec2-base fine-tune** (`voice_classifier/finetune_w2v.py`) — the production model

ROCm PyTorch exposes the GPU through the standard `torch.cuda` API, so the code runs unmodified.

**Minimal qualifying run** (no huge downloads): steps 1–4 train on LibriSpeech + edge-tts (~2 GB, ~30 min total).
**Full run** adds the XTTS-v2 clone set and the In-the-Wild dataset (step 5, optional, +9 GB) for the 4-tier evaluation.

In [ ]:
# 1. Environment check — should print an AMD GPU (e.g. 'AMD Instinct MI300X') and is ROCm: True
import torch
print('torch      :', torch.__version__)
print('gpu ok     :', torch.cuda.is_available())
print('device     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('is ROCm    :', torch.version.hip is not None, '| hip:', torch.version.hip)

In [ ]:
# If torch is missing or CPU-only on this AMD machine, install ROCm wheels first:
# %pip install torch torchaudio --index-url https://download.pytorch.org/whl/rocm6.2
%pip install -q soundfile librosa edge-tts tqdm scikit-learn transformers

In [ ]:
# 2. Get the code (skip if you uploaded the repo manually)
# !git clone https://github.com/Vishalkagade/amd-hackthaon.git
# %cd amd-hackthaon
import os
print('working dir:', os.getcwd())
assert os.path.exists('voice_classifier/train.py'), 'run this notebook from the repo root'

In [ ]:
# 3. Build the base dataset: LibriSpeech (human) + edge-tts neural voices (AI)
#    ~15-30 min depending on network. Idempotent — safe to re-run.
!python -m voice_classifier.prepare_data --max-human 1200 --max-ai 1200

In [ ]:
# 4. Train BOTH detectors on the AMD GPU. Checkpoints + logs land in voice_trends/
!python -m voice_classifier.train --epochs 14 --batch-size 64 --out voice_trends
!python -m voice_classifier.finetune_w2v --epochs 5

In [ ]:
# 5. OPTIONAL full pipeline — clone attack set + real-world deepfakes (+~9 GB):
#    5a. In-the-Wild dataset (real internet deepfakes):
# !curl -sL -o data_raw/itw.zip 'https://owncloud.fraunhofer.de/index.php/s/JZgXh0JEAF0elxa/download' && cd data_raw && unzip -q itw.zip && rm itw.zip
#    5b. XTTS-v2 clones need Coqui TTS in a SEPARATE venv (it pins transformers 4.x / torch 2.8):
# !python3 -m venv .venv-tts && .venv-tts/bin/pip install 'coqui-tts[codec]' 'transformers<5' 'torch==2.8.*' 'torchaudio==2.8.*' librosa soundfile tqdm
# !COQUI_TOS_AGREED=1 .venv-tts/bin/python -m voice_classifier.make_xtts_clones --n 350
# !COQUI_TOS_AGREED=1 .venv-tts/bin/python -m voice_classifier.make_xtts_clones --itw-refs --n 120
#    then re-run step 4, and evaluate:
# !python -m voice_classifier.eval_deepfake

In [ ]:
# 6. Verify: logs record the GPU they trained on — this is the AMD-usage evidence
import json
for log_path in ('voice_trends/training_log.json',
                 'voice_trends/wav2vec2_finetuned/finetune_log.json'):
    log = json.load(open(log_path))
    print(f"{log_path}\n  trained on: {log['gpu']}  torch: {log['torch']}"
          f"  rocm: {log.get('rocm')}  best: {log['best_val_acc']:.4f}")

In [ ]:
# 7. Sanity inference: one human + one AI sample through the production detector
from voice_classifier.infer import load_best_detector
from pathlib import Path
det, name = load_best_detector()
print('using:', name)
h = next(Path('data/human').glob('*.wav')); a = next(Path('data/ai').glob('*.wav'))
print('human clip -> ai_prob =', round(det.score_file(str(h))[0], 3))
print('ai clip    -> ai_prob =', round(det.score_file(str(a))[0], 3))